In this chapter is created to explain how create the pipiline function and how to use it well, also you're gonna find information about how the token function is implement

In [7]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [2]:
# en esta seccion vamos a entender mejor que es lo que pasa debajo del pipeline 
from transformers import pipeline 

classifier = pipeline("sentiment-analysis")
classifier(
    [
        "I've been waiting for a hugginFace course my whole life.",
        "I hate this so much!"
    ]
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9941714406013489},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

the last process has three steps 

1: preprocessing
2: passing the inputs throught the model 
3: postporcessing 

# Understanding the preprocessing 

---

## What "preprocessing" means here 

in classic Ml, preprocessing micht mean:

1.lowercasing 
2.removing stopwords
3.stemming
4. hand-creafted features 

**transformers do not work like that.**

in transformers, preporcessing = tokenization 


> **The preporcessing must be identical to the preprocessing used during pretraining.**

when a model is pretrained, Hugging Face saves two inseparable things: 

### the model weights 

this are the parameters that neural network will need 

### the tokenizer configuration 

this includes: 

vocabulary (vocab.txt / merges.txt)

tokenization rules (WordPiece, BPE, etc.)

lowercase or not

special tokens ([CLS], [SEP], [PAD])

max length

padding & truncation rules

In [3]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint) 

# la funcion checkpoint busca en el hub el id y descarga los pesos del modelo 
# y su configuracion 
# la funcion from_pretrained nos permite realizar esto ultimo 



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [5]:
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!"
]

inputs = tokenizer(raw_inputs,padding=True,truncation=True,return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


### what the parameters inside the tokenizer make?

tokenizer(raw_inputs,padding=True,truncation=True,return_tensors="pt")

this line turns human text into model-ready tensors. 

the tokenizer does three jobs at once: 

split text -> tokens
converttokens -> numbers 
package everything so the model can consume it 

#### padding = true

here solve this problem: 

your two sentences have different lenghts. 

models need rectangular tensors, not jaggled list. 

so the shorter sequences are filled with a special token: 

[PAD]

example:

sentences 1: 12 tokens 
sentences 2: 6 tokens 

after the padding the model make this 
sentence 1: 12 tokens 
sentences 2: 6 tokens + 6 [PAD] 

when we say they need to be rectangular it mean 

sentences 1: [101,2052,2003,102]
sentences 2: [101,2023,102]

this is not posible 

so with padding true with make this 
Sentence 1: [101, 2054, 2003, 102]
Sentence 2: [101, 2023, 102,   0]

it at some values to make the tokens form a rectangular form 

#### trucation = true 

makes this: tokens beyond the model's maximum lenght are eliminated 

so if the model has 1000 tokens per input and an input has 1200 the model cut it into 992 so we lost values of that token 


In [6]:
from transformers import AutoModel

# definimos el checkpoint 
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

### What is a "hight-dimensional-vector"

A hight-dimensional-vector is vector that the dimension of this is too big 
and it has hundreds or thousands of numbers 

In [ ]:
import torch

torch.Size([2,16,768])

# this what the model could return into us 

# so let's to see what each value on the tensor mean 

# the first value mean the number of sentences we passed 
# this the number of tokens after padding/truncation 
# the 3 value is the most important because say us each token is represented as a vector of 768 numbers

torch.Size([2, 16, 768])

In [10]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape) # this going to show us the structure of the dimension of the vector 

torch.Size([2, 16, 768])


In [12]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

In [ ]:
# for make the post processint the ouput we 
print(outputs.logits) # it is going to print the predicted but this are only log 

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


In [14]:
# to see the really predict into % values 
predictions = torch.nn.functional.softmax(outputs.logits,dim=-1)
print(predictions)

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


In [15]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}